# Tahoe Regional Transportation Plan Forecasting
> Data Engineering Tasks
* Residential development forecasting for 2035 and 2050

## Setup

In [ ]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import arcpy
from utils import *
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine

# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory TravelDemandModel\2022\
#data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data/base_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = out_dir / 'attributed_parcels.geoparquet'


# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY']

### Get Data

In [ ]:

# read in travel demand model base year parcel data - CHANGE With new name
parcel_base_tdm  = r"Base\data\processed_data\parcels_base_year.parquet"
parcel_base_path = local_path.parents[1].joinpath(parcel_base_tdm)
# read in base year parcel data
sdf_parcel_base  = pd.read_parquet(parcel_base_path)
#output_gdb = r"C:\GIS\Scratch.gdb"
# parcel development layer polygons

# CLEAN THIS UP - We should just populate this in the base year parcel data

# get parcel level data from LTinfo
dfIPES       = pd.read_json("https://www.laketahoeinfo.org/WebServices/GetParcelIPESScores/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476")
dfLCV_LTinfo = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetParcelsByLandCapability/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfRetired    = pd.read_json("https://www.laketahoeinfo.org/WebServices/GetAllParcels/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476")
dfBankedDev  = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetBankedDevelopmentRights/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfTransacted = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetTransactedAndBankedDevelopmentRights/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfAllParcels = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetAllParcels/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')

# get use tables 
# zoning data
sde_engine = get_conn('sde')
with sde_engine.begin() as conn:
    df_uses    = pd.read_sql("SELECT * FROM sde.SDE.PermissibleUses", conn)
    df_special = pd.read_sql("SELECT * FROM sde.SDE.Special_Designation", conn)
    df_parcel = read_sql_no_geom("SELECT * FROM sde.SDE.Parcel_History_Attributed WHERE YEAR = 2022", conn)

c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\arcgis\features\geo\_io\fileops.py:893: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=df_fields)
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\arcgis\features\geo\_io\fileops.py:893: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or a

### Parcel Data Engineering

In [ ]:
# Work from sdf_parcel_base since that's our current base year scenario
# This is just using the fully attributed parcel history layer to update fields in the sdf_parcel_base which is missing some data for the base year (2022)
update_fields = ['APO_ADDRESS', 'YEAR', 'COUNTY_LANDUSE_DESCRIPTION', 'REGIONAL_LANDUSE', 'YEAR_BUILT', 'PLAN_ID', 'PLAN_NAME', 'ZONING_ID', 'ZONING_DESCRIPTION', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'WITHIN_BONUSUNIT_BNDY']
for field in update_fields:
    sdf_parcel_base[field] = sdf_parcel_base.APN.map(dict(zip(df_parcel.APN, df_parcel[field]))) 

In [ ]:
# spatial dataframe with only initial columns
sdfParcels = sdf_parcel_base[initial_columns]

# map dictionary to sdf_units dataframe to fill in TAZ and Block Group fields
#sdfParcels['TAZ']                   = sdfParcels.APN.map(dict(zip(sdf_units_taz.APN,   sdf_units_taz.TAZ_1)))
#sdfParcels['BLOCK_GROUP']           = sdfParcels.APN.map(dict(zip(sdf_units_block.APN, sdf_units_block.TRPAID)))
# map IPES score to parcels
sdfParcels['IPES_SCORE']            = sdfParcels['APN'].map(dict(zip(dfIPES.APN, dfIPES.IPESScore)))
sdfParcels['IPES_SCORE_TYPE']       = sdfParcels['APN'].map(dict(zip(dfIPES.APN, dfIPES.IPESScoreType)))
# retired parcels
sdfParcels['RETIRED']               = sdfParcels['APN'].map(dict(zip(dfAllParcels.APN, dfAllParcels.RetiredFromDevelopment)))
# define housnig zoning and density
sdfParcels['HOUSING_ZONING']          = 'NA'
sdfParcels['COMMERCIAL_ALLOWED']      = 'No'
sdfParcels['TOURIST_ALLOWED']         = 'No'

# if the zoning id is in the list of multiple family zones then set to MF
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_sf_mf_zones(df_uses)['Zoning_ID']), 'HOUSING_ZONING'] = 'SF/MF'
# if the zoning id is in the list of single family zones and not in the multiple family zones then set to SF only
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_sf_only_zones(df_uses)['Zoning_ID']), 'HOUSING_ZONING'] = 'SF_only'
# if the zoning id is in the list of multiple family zones and not in the single family zones then set to MF only
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_mf_only_zones(df_uses)['Zoning_ID']), 'HOUSING_ZONING'] = 'MF_only'
# if the zoning id is in the list of commercial zones then set to Commercial
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_commercial_zones(df_uses)['Zoning_ID']), 'COMMERCIAL_ALLOWED'] = 'Yes'
# if the zoning id is in the list of tourist zones then set to Tourist Accommodation
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_tourist_zones(df_uses)['Zoning_ID']), 'TOURIST_ALLOWED'] = 'Yes'

# if COUNTY is in EL or PL or WA and SF allowed then set ADU_ALLOWED to yes or if COUNTY is in WA, DG, or CC and parcel acres is greater than 1 and SF allowed then set ADU_ALLOWED to yes
sdfParcels['ADU_ALLOWED'] = 'No'
sdfParcels.loc[(sdfParcels['COUNTY'].isin(['EL','PL', 'WA'])) & (~sdfParcels['HOUSING_ZONING'].isin(['MF_only', 'NA'])), 'ADU_ALLOWED'] = 'Yes'
sdfParcels.loc[(sdfParcels['COUNTY'].isin(['WA','DG','CC'])) & (sdfParcels['PARCEL_ACRES']>=1) &(~sdfParcels['HOUSING_ZONING'].isin(['MF_only', 'NA'])), 'ADU_ALLOWED'] = 'Yes'

# get density for MF and MF only zones, max residential units, and adjusted residential units
dfMF = get_mf_zones(df_uses)
sdfParcels['DENSITY']                    = sdfParcels['ZONING_ID'].map(dict(zip(dfMF.Zoning_ID, dfMF.Density)))
sdfParcels['MAX_RESIDENTIAL_UNITS']      = sdfParcels['PARCEL_ACRES'] * sdfParcels['DENSITY']
sdfParcels['MAX_UNITS']                  = sdfParcels['MAX_RESIDENTIAL_UNITS']*0.6
sdfParcels['MAX_UNITS']                  = sdfParcels['MAX_UNITS'].fillna(0).astype(int)

# set SF only zones to 1 max unit
sdfParcels.loc[sdfParcels['HOUSING_ZONING'] == 'SF_only', 'MAX_UNITS'] = 1

# set field for underbuilt evaluation
sdfParcels['POTENTIAL_UNITS'] = 0
sdfParcels['POTENTIAL_UNITS'] = sdfParcels['MAX_UNITS'] - sdfParcels['Residential_Units']
# set negative values to 0
sdfParcels.loc[sdfParcels['POTENTIAL_UNITS'] < 0, 'POTENTIAL_UNITS'] = 0

# calculate parcels with the greatest buildable potential  filter to the top 10% of parcels
# what value is in the top 10% of the potential buildable units
top_10_threshold = sdfParcels.POTENTIAL_UNITS.quantile(0.9)
# filter out rows where POTENTIAL_BUILDABLE_UNITS is NaN
sdfParcels['TOP_TEN_POTENTIAL_UNITS'] = sdfParcels.apply(lambda x: 'Yes' if x['POTENTIAL_UNITS'] >= top_10_threshold else 'No', axis=1)

# set FORECASTED_RESIDENTIAL_UNITS to 0
sdfParcels['FORECASTED_RESIDENTIAL_UNITS']     = 0
# set FORECAST_COMMERCIAN_UNITS to 0
sdfParcels['FORECASTED_COMMERCIAL_SQFT']       = 0
# set FORECAST_TOURIST_UNITS to 0
sdfParcels['FORECASTED_TOURIST_UNITS']         = 0
# set FORECAST_REASON to na
sdfParcels['FORECAST_REASON']                  = None
# FORECASTED_OCCUPANCY_RATE as a float field
sdfParcels['FORECASTED_RES_OCCUPANCY_RATE']    = 0.0


sdfParcels.to_parquet(parcel_pickle_part1)

Exception: Spatial column not defined, please use `set_geometry`

## Model Year Forecasting

### Residential Forecasting

> Methods

1) Assign Known Projects - No longer relevant since this is from 2025
* Assign using Lookup_Lists\forecast_residential_assigned_units.csv: 
    * Known Residential Allocations from 2023-2024, 
    * Known Residential Bonus Units from permitted projects, 
    * applications in review, and 
    * RBU reservations in LT Info
    * Known Accessory Dwelling permits not completed
    * Remove Known Banking projects that removed units in 2023-2024
* Calcuate Remaining local jurisdiction pool units less units used for known projects
    
2) Assign Residential Bonus Units within Bonus Unit Boundary
* Full build out of CTC Asset Lands using jurisdiction bonus unit pools
* Identify vacant buildable lots within Bonus Unit boundary
* Assign remaining jurisdiction pool units to available parcels within jurisidiction

3) Assigne remaining Jurisdiction Residential Bonus and General Units
* Identify	Vacant buildable lots with allowed Multi-family use, calculate allowed density
* Assign 15% of remaining local jurisdiction Residential Allocation pool units to available multi-family parcels within jurisidiction
* Assign 35% of remaining Banked units to available multi-family parcels (use adjusted weighting from existing residential units?)
* Assign 35% of remaining Converted units to available multi-family parcels (use adjusted weighting from existing residential units?)
    
4) Assign remaining TRPA pool units to available parcels throughout region (use adjusted weighting from existing residential units?)
* Evaluate Vacant Buildable Lots with Single-family Residential Allowed Use	
* Identify	Vacant buildable lots with allowed Single-family use
* Identify	Accessory Dwelling Uses Allowed (All California Parcels and NV Parcels Greater than 1 Acre)
* Evaluate Underbuilt parcels with Multi-family Residential Allowed Use	
* Identify	Underbuilt Residential lots with allowed Multi-family use
* Evaluate Underbuilt parcels with Accessory Dwelling Uses Allowed (All California Parcels and NV Parcels Greater than 1 Acre)	
* Identify parcels that are in Town Centers or within a quarter mile and are top x% of underbuilt parcels. 
* Underbuilt = exclude existing condo and common area land uses. sf mf or mixed use
* ADU potential = exclude existing condo or common area land uses, and wait to use any NV parcels>acre. 


> Get the Parcel Base Data

In [ ]:
# get pickle 1 as a spatially enabled dataframe - spatial joins, foreign keys, and new categorical fields added already
sdfParcels = pd.from_pickle(parcel_pickle_part1)
# Randomly sort parcels so that development can be assigned randomly
sdfParcels = sdfParcels.sample(frac=1).reset_index(drop=True)
# Export index and apn to a csv file for future reference with a name that includes the date and time
# # This will allow us to recreate the same random order in the future
sdfParcels[['APN']].to_csv(f"Parcel_Sort_Order_{pd.Timestamp.now().strftime('%Y%m%d%H%M%S')}.csv", index=True)

> Get Lookup Lists

In [ ]:
# path to lookup lists
res_assigned_lookup = "Lookup_Lists/known_projects.csv"
res_zoned_lookup    = "Lookup_Lists/forecast_residential_zoned_units.csv"
#ctc_assetlands_lookup = "Lookup_Lists/CTC_AssetLands_Lookup.csv"

# get zoned and assigned lookup lists as data frames
dfResZoned    = pd.read_csv(res_zoned_lookup)
dfResAssigned = pd.read_csv(res_assigned_lookup)
# get lookup list for CTC asset lands (17 parcels)
#ctc_parcels = pd.read_csv(ctc_assetlands_lookup)

> Assign Known Projects

In [ ]:
# Forecast Known Residential Projects from 2023-2024
# group dfResAssigned by APN and sum Unit Change to aggregate to one total for duplciate
dfResAssigned = dfResAssigned[~(dfResAssigned['Development Right'] == 'Commercial Floor Area (CFA)')]
dfResAssigned = dfResAssigned[~(dfResAssigned['Development Right'] == 'Tourist Accommodation Unit (TAU)')]
dfResAssigned['Quantity'] = dfResAssigned['Quantity'].fillna(0).astype(int)
dfResAssignedGrouped_APN = dfResAssigned.groupby('APN', as_index=False).agg({'Quantity': 'sum'})
# # assign forecast residential units for assigned projects
sdfParcels['FORECASTED_RESIDENTIAL_UNITS'] = sdfParcels.APN.map(dict(zip(dfResAssignedGrouped_APN.APN, dfResAssignedGrouped_APN['Quantity'])))
# # forecast reason = Assigned for APNs in dfResAssignedGrouped
sdfParcels.loc[sdfParcels['APN'].isin(dfResAssigned['APN']), 'FORECAST_REASON'] = 'Assigned'

In [ ]:
sdfParcels.groupby(['JURISDICTION','FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum()

In [ ]:
# total forecasted units
total_forecasted_units = sdfParcels['FORECASTED_RESIDENTIAL_UNITS'].sum()
print(f'Total Forecasted Units: {total_forecasted_units}')

In [ ]:
import numpy as np


# South lake VHR conditions - why isn't this populated
VHR_condition = "(JURISDICTION == 'CSLT') & (Residential_Units > 0) & (VHR == 'No') & (TouristAccommodation_Units == 0)& (Residential_Units <3)"
CSLT_VHR_Eligble = sdfParcels.query(VHR_condition)
# Randomly assign New_VHR = 'Yes' to APNs until cumulative Residential_Units reaches 900


target_units = 900
shuffled = CSLT_VHR_Eligble.sample(frac=1, random_state=42).reset_index(drop=True)
cumulative = shuffled['Residential_Units'].cumsum()
selected_apns = shuffled.loc[cumulative <= target_units, 'APN']

sdfParcels['New_VHR'] = 'No'
sdfParcels.loc[sdfParcels['APN'].isin(selected_apns), 'New_VHR'] = 'Yes'

assigned_units = sdfParcels.loc[sdfParcels['New_VHR'] == 'Yes', 'Residential_Units'].sum()
print(f"Parcels assigned New_VHR = Yes: {sdfParcels['New_VHR'].eq('Yes').sum()}")
print(f"Total Residential_Units assigned: {assigned_units}")

In [ ]:
# Compute updated unit pool after assigned projects
dfGroup_Assigned = dfResAssigned.groupby(['Jurisdiction', 'Unit_Pool']).sum('Quantity').reset_index()
dfGroup_Assigned = dfGroup_Assigned.drop(columns=['Occupancy_Rate'])
dfGroup_Assigned = dfGroup_Assigned.rename(columns={'Quantity': 'Unit_Change'})
dfPool_Assigned = pd.merge(dfResZoned, dfGroup_Assigned, on=['Jurisdiction', 'Unit_Pool'], how='left')
# We will end up suing Unit_Change to adjust the number of units from 
# unit assignment csvs for each scenario
dfPool_Assigned['Unit_Change'] = dfPool_Assigned['Unit_Change'].fillna(0)
dfPool_Assigned['Future_Units_Adjusted'] = dfPool_Assigned['Future_Units'] - dfPool_Assigned['Unit_Change']
dfPool_Assigned

In [ ]:
# Export updated unit pool CSV and base parcel layer pickle
dfPool_Assigned.to_csv(out_dir / 'unit_pool_assigned.csv', index=False)
sdfParcels.to_pickle(out_dir / 'base_parcel_data.pkl')
print('Exports complete: unit_pool_assigned.csv and base_parcel_data.pkl')